Install Required Libraries

In [1]:
!pip install -q -U langchain langchain-google-genai langchain-community tavily-python requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


Import Libraries & Retrieve API Keys

In [2]:
import requests
from google.colab import userdata
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

TAVILY_API_KEY = userdata.get("tavily_apikey")
RAPIDAPI_KEY = userdata.get("rapid_apikey")
GOOGLE_API_KEY = userdata.get("gemini_api_key")

/tmp/ipykernel_1337/2113162969.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


Define Tools & System Prompt

In [3]:
skill_demand_tool = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=TAVILY_API_KEY,
)

@tool
def search_jobs(skill: str, location: str) -> list:
    """
    Search for jobs requiring a specific skill using the JSearch API.
    """
    print("\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")
    url = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key": RAPIDAPI_KEY,
        "x-rapidapi-host": "jsearch.p.rapidapi.com",
    }
    params = {
        "query": f"{skill} in {location}",
        "page": "1",
        "num_pages": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience",
    }
    response = requests.get(url, headers=headers, params=params)
    data = response.json()
    jobs = data.get("data", [])
    print(f"Found {len(jobs)} jobs\n")
    return [
        {
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link"),
        }
        for job in jobs
    ]

SYSTEM_PROMPT = """
You are a Skill-to-Career Mapping assistant that helps students understand skill demand
and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Research industry demand, salary insights, and career trends
- search_jobs: Find real job listings based on skills and location

Present results in a clean, readable format with clear sections and spacing.
Include all job details with apply links.
Do not use markdown formatting.
"""

/tmp/ipykernel_1337/1706373345.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  skill_demand_tool = TavilySearchResults(


Initialize Model, Agent, and Memory Checkpointer

In [4]:
model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=GOOGLE_API_KEY,
)

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[skill_demand_tool, search_jobs],
    checkpointer=checkpointer,
    debug=True,
)

Run First User Query (Skill Demand & Job Search)

In [8]:
user_query = (
    "What's the demand for generative AI in the industry "
    "and show me related job openings in India"
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": user_query}]},
    config=config,
)

print(response["messages"][-1].content[0]["text"])

[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='0320a169-f738-43c0-876c-c0d631b2286b'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_results_json', 'arguments': '{"query": "demand for generative AI in industry"}'}, '__gemini_function_call_thought_signatures__': {'8309a9d4-96e3-4f10-aaaa-1bb338922a5b': 'Cu0EARFNMg/A+SYCYjUP7pzhfYMlhKHwcnwz4uvq8Su5Q+1ToGqbm9R8hQLe0hUogb69cyDDx/LLjNQhN5FYzha0vFdNhV310ogeTsK/99QCBhFBQZKoVJZL7vYA5gZY80n0LYVAnk0aZlNFggB527sztIfFyRvZGyEcQuNuN81BNJ3FIVBqTfoKjchbzv6U4z4GusxAKQtfLpoIuAZhNJhOXoxqAISy98VJe4NFSZnQ5qIvpSHADpZhb4bmaCUpXAepkYacxs+FnNhR787GLfz1zjR6xfFeXiWkKtCtd3ZOUSxy55xk3NJ/XHUJICup78Dbh/LLOIpSfvELPFKAiUfyczYrcWI7NafgCo9GpaV/mK1VMutcdhzypiDg4zVQoGB3p4pvvT/dA/imdnPS7EGd1FbL/kbJd3JNBJ6IlsZGvl1t52W+zM+D3kDA+inkyD9FIPUnYoblJu0upUpot8Kzl0zt00cQIQblpIF+ouqSuAC+2k8IsIYCUGLhwb3r1A6qXY

Run Follow-up Query (Conversational Context)

In [9]:
user_query = "Tell me more about the second job you showed"

response = agent.invoke(
    {"messages": [{"role": "user", "content": user_query}]},
    config=config,
)

print(response["messages"][-1].content[0]["text"])

[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='0320a169-f738-43c0-876c-c0d631b2286b'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_results_json', 'arguments': '{"query": "demand for generative AI in industry"}'}, '__gemini_function_call_thought_signatures__': {'8309a9d4-96e3-4f10-aaaa-1bb338922a5b': 'Cu0EARFNMg/A+SYCYjUP7pzhfYMlhKHwcnwz4uvq8Su5Q+1ToGqbm9R8hQLe0hUogb69cyDDx/LLjNQhN5FYzha0vFdNhV310ogeTsK/99QCBhFBQZKoVJZL7vYA5gZY80n0LYVAnk0aZlNFggB527sztIfFyRvZGyEcQuNuN81BNJ3FIVBqTfoKjchbzv6U4z4GusxAKQtfLpoIuAZhNJhOXoxqAISy98VJe4NFSZnQ5qIvpSHADpZhb4bmaCUpXAepkYacxs+FnNhR787GLfz1zjR6xfFeXiWkKtCtd3ZOUSxy55xk3NJ/XHUJICup78Dbh/LLOIpSfvELPFKAiUfyczYrcWI7NafgCo9GpaV/mK1VMutcdhzypiDg4zVQoGB3p4pvvT/dA/imdnPS7EGd1FbL/kbJd3JNBJ6IlsZGvl1t52W+zM+D3kDA+inkyD9FIPUnYoblJu0upUpot8Kzl0zt00cQIQblpIF+ouqSuAC+2k8IsIYCUGLhwb3r1A6qXY